In [1]:
print("ok")

ok


In [1]:
%pwd

'c:\\Users\\vasan\\OneDrive\\Desktop\\chatbot\\Medical-Chatbot\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\vasan\\OneDrive\\Desktop\\chatbot\\Medical-Chatbot'

In [5]:
import os
print(os.getcwd())

c:\Users\vasan\OneDrive\Desktop\chatbot\Medical-Chatbot


In [6]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


c:\Users\vasan\anaconda3\envs\Medical-Chatbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
#extract text from pdf files
def load_pdf_files(data):
    loader=DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
        )
    documents=loader.load()#documents will store the pdf page by page
    print(len(documents))
    return documents

In [8]:
extracted_data=load_pdf_files("data")

637


In [9]:
#we are cleaning the data just using content and source thats it

from typing import List
from langchain.schema import Document
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document]=[]
    for doc in docs:
        src=doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
            )
        )
    return minimal_docs

In [10]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [11]:
#chunking the data
def text_split(minimal_docs):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk=text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [12]:
texts_chunk=text_split(minimal_docs)
print(f"Number of Chunks:{len(texts_chunk)}")

Number of Chunks:5859


In [ ]:
#using a pretrained model extracting from huggingface
from langchain.embeddings import HuggingFaceEmbeddings
def download_hugging_face_embeddings():
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    embeddings=HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings
embedding=download_hugging_face_embeddings()

C:\Users\vasan\AppData\Local\Temp\ipykernel_18504\2269385190.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(


In [16]:
embedding

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [33]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [34]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

#Make sure these keys are available globally for other libraries
os.environ["PINECONE_API_KEY"]=PINECONE_API_KEY
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [16]:
from pinecone import Pinecone
pinecone_api_key=PINECONE_API_KEY

pc=Pinecone(api_key=pinecone_api_key)

In [17]:
from pinecone import ServerlessSpec
index_name="medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )
    index=pc.Index(index_name)

In [ ]:
#Create library + place books
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    index_name=index_name,
    embedding=embedding,
)

In [ ]:
#Open existing library and read books
from langchain_pinecone import PineconeVectorStore
docsearch=PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [19]:
#if we want to add more data to the existing data to pine cone index_name
#again create a dumpy document
#do this
dswith=Document(
    page_content="",
    metadata={"source":"youtube"}
)
docsearch.add_documents(documents=[dswith])


['498e2cab-8015-4cf0-b575-40875a65c026']

In [20]:
retriever=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [31]:
retrieved_docs=retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='907e4b41-c954-44d5-95e1-e0967f917385', metadata={'source': '..\\data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='990c3192-f8a1-40b4-8370-0cb7b25f1058', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='e5257459-6dc1-4bda-9fe3-1c898e70c1a7', metadata={'source': '..\\data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26')]

In [22]:
from langchain_openai import ChatOpenAI
chatModel=ChatOpenAI(model="gpt-4o")

In [25]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [27]:
system_prompt= """
You are an Medical assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, say that you don't know.
Use three sentences maximum and keep the answer concise.

{context}
"""
prompt = ChatPromptTemplate.from_messages(
   [   
    ("system", system_prompt),
    ("human", "{input}"),
   ]
)

In [28]:
question_answer_chain=create_stuff_documents_chain(chatModel,prompt)
rag_chain=create_retrieval_chain(retriever,question_answer_chain)

In [ ]:
response=rag_chain.invoke({"input":"what is Acromegaly and gigantism?"})
print(response["answer"])